# LAVEE — 01 · YOLO12 training and ten-fold evaluation

Clean, output-free copy of the supplied **Flying_Object_Deadline_10Fold_18k_All_Techniques (1).ipynb**.
Its original training code cells are preserved unchanged. This notebook contains the training workflow;
use **02_Detection_Tracking_Dashboard.ipynb** to run saved models without training.

**Before running:** select a Colab GPU and edit the configuration cell's `DATA_ARCHIVE` or `DATA_FOLDER`
to your deduplicated combined YOLO dataset. Keep the original labels and verify the class IDs.
`IMAGE_SAMPLES_PER_CLASS=3000` selects unique images containing each class; multilabel overlap means
this is not necessarily exactly 18,000 images or an equal number of boxed objects per class.

**The saved experiment:** YOLO12n, image size 416, ten folds, five detector epochs per fold,
four head epochs, two cGAN epochs. `RUN_FINAL=False` and `RUN_END_TO_END_EACH_FOLD=False` are retained.
Fold 1 export is a demo package; completing CV does not create an all-development-data final model.
LSTM/BiLSTM use spatial tokens, not video trajectories. cGAN is used only for training augmentation.

Existing completed folds can resume when the dataset fingerprint and run configuration match.
Do not run this notebook solely to open the dashboard. The recorded run used image-level fallback
groups; it cannot establish independence between frames from the same original recording.


#  10-fold YOLO12 + five hybrid classifiers (six classes)

**Purpose:** A genuine 10-fold evaluation under a severe Colab T4 time constraint, with **new independent YOLO12 training in every fold** and all five hybrid heads trained in every fold. The original 55,409-image dataset is left unchanged; only a separate sampled view is used. No previous six-class checkpoint is reused across CV folds, preventing fold-model weight leakage.

**Important:** 5 detector epochs and 4 head epochs are an aggressive speed compromise, **not equivalent to the original 30/15-epoch study**. This is a short-budget research experiment; report observed outcomes only, including possible low accuracy. Approximate 5-6-hour target is not guaranteed. The saved folds are recoverable after disconnect.

**Leakage limitation:** If `GROUP_CSV_PATH` is missing, the notebook uses image-level grouping and cannot establish source/video independence. Do not claim source-disjoint CV without a verified group manifest. Synthetic cGAN images are used for training only, never for validation.

**Metrics:** Per-fold detector mAP50/mAP50-95 and 5 hybrid *real ground-truth crop* accuracy/F1 across all 10 folds. End-to-end IoU-matched comparison is performed for Fold 1 only; it is not a 10-fold end-to-end hybrid claim. Locked external test/final retraining is disabled because it adds compute.


## 1. Install dependencies and select Colab GPU (Runtime → Change runtime type → T4 GPU)

In [ ]:
%pip -q install -U ultralytics timm scikit-learn pandas matplotlib pyyaml pillow tqdm
import os,sys,random,json,math,csv,hashlib,shutil,zipfile,time,platform,collections
from pathlib import Path
import numpy as np,pandas as pd,torch,yaml
from PIL import Image,ImageOps
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import classification_report,confusion_matrix,precision_recall_fscore_support,accuracy_score
from ultralytics import YOLO
import timm,ultralytics,sklearn
assert torch.cuda.is_available(), 'GPU needed: Runtime > Change runtime type > T4 GPU'
DEVICE='cuda'
print('GPU:',torch.cuda.get_device_name(0))
print('torch',torch.__version__,'ultralytics',ultralytics.__version__,'timm',timm.__version__,'sklearn',sklearn.__version__)

## 2. Configuration — edit this cell, then run in order

In [ ]:
# Put a zipped copy of E:\AI project\DS\comb DSF in Google Drive, then set DATA_ARCHIVE.
# You can alternatively use a folder already visible under /content or Drive (DATA_FOLDER).
USE_DRIVE=True
DATA_ARCHIVE='/content/drive/MyDrive/comb_DSF.zip'  # EDIT: absolute .zip path, or '' to upload interactively
DATA_FOLDER=''                                  # EDIT: absolute extracted folder, or leave ''
OUTPUT_ROOT='/content/drive/MyDrive/FlyingObject_YOLO12_CombinedDSF'  # persisted progress
CLASS_NAMES_OVERRIDE=None  # e.g. ['airplane','bird',...] ONLY if no data.yaml; verify exact 0..5 IDs first
GROUP_CSV_PATH=''   # optional CSV: columns image,group (or file,group); HIGHLY recommended for video/source grouping
REQUIRE_SOURCE_GROUPS=False # True: refuse image-level folds if group CSV unavailable

SEED=42
IMAGE_SAMPLES_PER_CLASS=3000  # min unique images CONTAINING each class, multi-label overlap allowed
EXTRA_BACKGROUND_IMAGES=200
RUN_FOLD1_E2E_PREVIEW=True   # optional ~1-3 minutes after CV finishes
NUM_FOLDS=10
YOLO_WEIGHTS='yolo12n.pt'  # actual YOLO12; can use 'yolo12s.pt' with more GPU resources
YOLO_IMGSZ=416
YOLO_EPOCHS=5
YOLO_BATCH=16
YOLO_PATIENCE=4
HEAD_EPOCHS=4
HEAD_BATCH=256
HEAD_LR=1e-3
GAN_EPOCHS=2
GAN_STEPS_PER_EPOCH=25
GAN_BATCH=32
GAN_MAX_NEW_PER_CLASS=100  # cap for T4 safety; set None ONLY if enough storage/RAM for exact target balancing
GAN_TARGET_QUANTILE=1.0  # aim for majority-class size; cap above can prevent equality
DETECT_CONF=0.25
MATCH_IOU=0.50

# Deadline full 10-fold experiment. Do not enable SMOKE_TEST for final CV.
SMOKE_TEST=False
RUN_FINAL=False
# Optional: skip computationally expensive end-to-end validation per fold (crop scores remain).
RUN_END_TO_END_EACH_FOLD=False

if SMOKE_TEST:
    YOLO_EPOCHS=2; HEAD_EPOCHS=2; GAN_EPOCHS=1; GAN_STEPS_PER_EPOCH=5
    RUN_FINAL=False

random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
OUT=Path(OUTPUT_ROOT);OUT.mkdir(parents=True,exist_ok=True)
print('Output directory:',OUT)

## 3. Import the deduplicated combined dataset — never use `balanced_six_class`

In [ ]:
def safe_extract(zip_path,dest):
    dest=Path(dest).resolve();dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        for m in z.infolist():
            target=(dest/m.filename).resolve()
            if not (target==dest or dest in target.parents): raise RuntimeError('Unsafe zip member: '+m.filename)
        z.extractall(dest)

if not DATA_FOLDER:
    if not DATA_ARCHIVE:
        from google.colab import files
        uploaded=files.upload()
        archives=[x for x in uploaded if x.lower().endswith('.zip')]
        assert len(archives)==1,'Upload exactly one ZIP archive, the deduplicated comb DSF dataset.'
        DATA_ARCHIVE='/content/'+archives[0]
    assert Path(DATA_ARCHIVE).is_file(),f'Archive missing: {DATA_ARCHIVE}. Update the configuration cell.'
    DATA_FOLDER='/content/combined_dsf_extracted'
    sentinel=Path(DATA_FOLDER)/'.extraction_complete'
    if not sentinel.exists():
        safe_extract(DATA_ARCHIVE,DATA_FOLDER)
        sentinel.write_text(str(DATA_ARCHIVE))
    else:
        assert sentinel.read_text()==str(DATA_ARCHIVE),'Extraction directory belongs to another archive. Delete extraction directory or edit DATA_FOLDER.'
raw_root=Path(DATA_FOLDER).resolve()
assert raw_root.is_dir(),f'DATA_FOLDER not found: {raw_root}'
assert 'balanced_six_class' not in str(raw_root).lower(),'Wrong archive: this notebook requires comb DSF AFTER deduplication.'
print('Dataset folder:',raw_root)
print('Sample entries:',list(raw_root.iterdir())[:12])

## 4. Locate YOLO-format image/label pairs and audit ALL six classes

In [ ]:
IMAGE_EXT={'.jpg','.jpeg','.png','.bmp','.webp'}
EXPECTED=set(['airplane','bird','helicopter','kite','parachute','uav'])
yamls=sorted([p for p in raw_root.rglob('*.yaml') if p.name.lower() in ('data.yaml','dataset.yaml','data.yml','dataset.yml')],key=lambda p:len(p.parts))
configs=[]
for yp in yamls:
    try:
        conf=yaml.safe_load(yp.read_text(encoding='utf-8-sig')) or {}
        if isinstance(conf,dict) and 'names' in conf:
            names=conf['names']
            if isinstance(names,dict): names=[names.get(i,names.get(str(i))) for i in range(len(names))]
            if len(names)==6 and all(isinstance(n,str) for n in names):configs.append((yp,[str(n).strip() for n in names]))
    except Exception as e:print('Skipping malformed YAML',yp,e)
if CLASS_NAMES_OVERRIDE is not None:
    CLASS_NAMES=list(CLASS_NAMES_OVERRIDE)
    print('WARNING: USING MANUAL CLASS IDS. Verify against source annotation mapping.')
elif configs:
    CLASS_NAMES=configs[0][1]
    for yp,names in configs:
        if [s.lower() for s in names] != [s.lower() for s in CLASS_NAMES]:
            raise ValueError(f'Conflicting class orders: {configs[0][0]} and {yp}; manually resolve before continuing')
    print('Verified class mapping from:',configs[0][0])
else:
    raise RuntimeError('No six-class data.yaml found. Inspect original annotation IDs, then set CLASS_NAMES_OVERRIDE=[class 0,...,class 5]. Do not guess mapping.')
assert len(CLASS_NAMES)==6 and set(x.lower() for x in CLASS_NAMES)==EXPECTED,CLASS_NAMES
print('Class ID mapping:',dict(enumerate(CLASS_NAMES)))

# Ultralytics finds each text annotation by replacing a path component 'images' with 'labels'.
# Preserve the ORIGINAL deduplicated dataset; no automatic label rewriting.
def label_for(p):
    parts=list(p.parts)
    k=max((i for i,x in enumerate(parts) if x.lower()=='images'),default=-1)
    if k<0:return None
    parts[k]='labels'
    return Path(*parts).with_suffix('.txt')
images=[p for p in raw_root.rglob('*') if p.suffix.lower() in IMAGE_EXT and 'images' in [q.lower() for q in p.parts]]
images=sorted(set(p.resolve() for p in images))
assert images, 'No YOLO images/ directory found. Required: images/... and mirrored labels/...'

def read_annots(label_path):
    if not label_path.exists():return [] # YOLO convention: missing label means background
    result=[]
    for line_no,line in enumerate(label_path.read_text(encoding='utf-8-sig').splitlines(),1):
        if not line.strip():continue
        fields=line.split()
        if len(fields)!=5:raise ValueError(f'Invalid label format {label_path}:{line_no}: {line}')
        try: values=[float(v) for v in fields]
        except ValueError as e:raise ValueError(f'Invalid numeric label: {label_path}:{line_no}') from e
        cls,x,y,w,h=values
        if cls!=int(cls) or not (0<=cls<6):raise ValueError(f'Class ID out of range: {label_path}:{line_no}: {cls}')
        if not(0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1 and x-w/2>=-1e-3 and y-h/2>=-1e-3 and x+w/2<=1+1e-3 and y+h/2<=1+1e-3):
            raise ValueError(f'Invalid YOLO box: {label_path}:{line_no}: {values}')
        result.append((int(cls),x,y,w,h))
    return result

records=[]; objects=[]
for ip in tqdm(images,desc='Parsing real deduplicated annotations'):
    lp=label_for(ip);ann=read_annots(lp)
    idx=len(records)
    records.append({'idx':idx,'image':str(ip),'relative':str(ip.relative_to(raw_root)),'label':str(lp),'n_objects':len(ann)})
    for cls,x,y,w,h in ann:
        objects.append({'image_idx':idx,'class_id':cls,'bbox':(x,y,w,h)})
assert len(records)>0 and len(objects)>0
counts=collections.Counter(obj['class_id'] for obj in objects)
assert all(counts.get(j,0)>0 for j in range(6)),f'Class missing: {counts}'
audit={'original_reference_retained_images':55409,'original_reference_annotated_objects':86254,'original_reference_background_images':1174,
       'actual_images':len(records),'actual_annotations':len(objects),
       'actual_background_images':sum(r['n_objects']==0 for r in records),
       'classes':CLASS_NAMES,'counts_by_class':{CLASS_NAMES[j]:counts[j] for j in range(6)}}
print(json.dumps(audit,indent=2))
(OUT/'dataset_audit.json').write_text(json.dumps(audit,indent=2))
if len(records)!=55409 or len(objects)!=86254 or audit['actual_background_images']!=1174:
    print('AUDIT WARNING: Actual counts differ from previously reported canonical audit. Investigate before citing figures.')

# Deadline-only sampling: at least 3,000 images CONTAINING each class.
# Multi-label images count for each class they genuinely contain; no duplicate files.
# Original image files and ALL their annotated bounding boxes remain unchanged.
full_image_records=records
full_objects=objects
full_image_classsets=[set() for _ in full_image_records]
for ob in full_objects: full_image_classsets[ob['image_idx']].add(ob['class_id'])
class_image_ids={c:[i for i,cs in enumerate(full_image_classsets) if c in cs] for c in range(6)}
for c,pool in class_image_ids.items():
    assert len(pool)>=IMAGE_SAMPLES_PER_CLASS, (
        f'{CLASS_NAMES[c]} has only {len(pool)} unique source images, '
        f'below requested {IMAGE_SAMPLES_PER_CLASS}. Lower limit and rerun.')
background_bucket=[i for i,cs in enumerate(full_image_classsets) if not cs]
rng=np.random.default_rng(SEED)
selected=set()
# Select rarer classes first so shared multi-label images help meet multiple quotas.
for c in sorted(range(6),key=lambda k:len(class_image_ids[k])):
    already=sum(c in full_image_classsets[i] for i in selected)
    needed=max(0,IMAGE_SAMPLES_PER_CLASS-already)
    remaining=[i for i in class_image_ids[c] if i not in selected]
    if needed:
        assert len(remaining)>=needed
        selected.update(map(int,rng.choice(remaining,size=needed,replace=False)))
    print(CLASS_NAMES[c],':',sum(c in full_image_classsets[i] for i in selected),'images selected containing class')
if background_bucket and EXTRA_BACKGROUND_IMAGES>0:
    selected.update(map(int,rng.choice(background_bucket,size=min(EXTRA_BACKGROUND_IMAGES,len(background_bucket)),replace=False)))
chosen=sorted(selected)
index_remap={int(old):new for new,old in enumerate(chosen)}
records=[dict(full_image_records[old],idx=new) for new,old in enumerate(chosen)]
objects=[dict(ob,image_idx=index_remap[ob['image_idx']]) for ob in full_objects if ob['image_idx'] in index_remap]
counts=collections.Counter(o['class_id'] for o in objects)
assert len(records)==len(chosen) and len(set(r['image'] for r in records))==len(records)
assert all(counts.get(c,0)>0 for c in range(6))
subset_audit={
 'selection_method':'quota of unique images containing each class, with multi-label overlap; all original boxes retained',
 'source_images_total':len(full_image_records),'source_boxes_total':len(full_objects),
 'selected_images':len(records),'selected_boxes':len(objects),
 'selected_background_images':sum(r['n_objects']==0 for r in records),
 'minimum_images_with_each_class':IMAGE_SAMPLES_PER_CLASS,
 'images_with_class':{CLASS_NAMES[c]:sum(c in full_image_classsets[z] for z in chosen) for c in range(6)},
 'annotations_by_class':{CLASS_NAMES[c]:counts[c] for c in range(6)},
 'original_indices':chosen}
(OUT/'deadline_subset_audit.json').write_text(json.dumps(subset_audit,indent=2))
print('Deadline subset images:',len(records),'boxes:',len(objects))
print('Class image counts:',subset_audit['images_with_class'])
print('Actual boxed-object counts:',subset_audit['annotations_by_class'])
del full_image_records,full_objects,full_image_classsets


## 5. Dataset distributions (actual annotated objects, before any augmentation)

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,4))
ax.bar(CLASS_NAMES,[counts[i] for i in range(6)])
ax.set(title='Deduplicated combined dataset — real object counts',ylabel='Annotated object count')
plt.xticks(rotation=15);plt.tight_layout()
fig.savefig(OUT/'original_class_counts.png',dpi=160)
plt.show()

## 6. Source groups and leakage-aware locked test split + 10 folds

**Best practice:** supply `GROUP_CSV_PATH` with `image,group` for original video/sequence/source IDs. **Never** use class IDs as groups or split individual frames from the same sequence across train/test. If no metadata is available, the notebook honestly marks results as **image-level folds (source grouping unknown)**, not source-independent validation. Group leakage cannot be eliminated just because exact image duplicates were removed.

Each image may contain multiple classes. The splitter uses the **rarest present class** as a *proxy* for stratification; this is not full multilabel stratification, so the code explicitly checks every class in every fold and refuses invalid 10-fold partitions. Negatives receive proxy class 6.

In [ ]:
group_map={}
if GROUP_CSV_PATH:
    gp=Path(GROUP_CSV_PATH)
    assert gp.is_file(),f'Group file not found: {gp}'
else:
    candidates=[p for p in raw_root.rglob('*.csv') if ('manifest' in p.name.lower() or 'group' in p.name.lower())]
    gp=candidates[0] if candidates else None
if gp:
    try:
        gg=pd.read_csv(gp)
        file_col=next((k for k in ('image','file','filename','path','image_path') if k in gg.columns),None)
        group_col=next((k for k in ('group','group_id','source_group','sequence_id') if k in gg.columns),None)
        if file_col and group_col:
            group_map={str(k).replace('\\','/').lstrip('./'):str(v) for k,v in zip(gg[file_col],gg[group_col]) if pd.notna(v)}
            print('Read source group metadata from:',gp)
    except Exception as exc: print('Could not use group CSV:',exc)

def lookup_group(rec):
    rel=rec['relative'].replace('\\','/')
    for key in (rel,Path(rel).name,Path(rel).stem):
        if key in group_map:return group_map[key]
    return None
source_groups=[lookup_group(r) for r in records]
if all(x is not None for x in source_groups):
    GROUP_BASIS='source_metadata'
    groups=np.asarray(source_groups,dtype=object)
else:
    if REQUIRE_SOURCE_GROUPS:raise RuntimeError('Source group information missing. Provide GROUP_CSV_PATH for credible source-group CV.')
    GROUP_BASIS='image_level_fallback_source_unknown'
    groups=np.array(['image_'+str(i) for i in range(len(records))],dtype=object)
    print('IMPORTANT LIMITATION: no complete source-group manifest; image-level folds may share related frames/sources.')

image_classes=[set() for _ in records]
for ob in objects:image_classes[ob['image_idx']].add(ob['class_id'])
proxies=np.array([min(cs,key=lambda c:(counts[c],c)) if cs else 6 for cs in image_classes],dtype=np.int64)
index=np.arange(len(records))
# Outer test: 1/10 of original real images, never used in CV / tuning.
outer=list(StratifiedGroupKFold(n_splits=10,shuffle=True,random_state=SEED).split(index,proxies,groups))
valid_outer=[]
for dev_idx,test_idx in outer:
    if all(any(c in image_classes[i] for i in test_idx) and any(c in image_classes[i] for i in dev_idx) for c in range(6)):
        valid_outer.append((dev_idx,test_idx))
assert valid_outer,'Cannot obtain 10% group-held test with all classes. Improve grouping/coverage; do not silently drop classes.'
dev_idx,test_idx=valid_outer[0]
cv=list(StratifiedGroupKFold(n_splits=NUM_FOLDS,shuffle=True,random_state=SEED+1).split(dev_idx,proxies[dev_idx],groups[dev_idx]))
folds=[]
for j,(a,b) in enumerate(cv,1):
    train_ids=dev_idx[a];val_ids=dev_idx[b]
    assert not (set(groups[train_ids])&set(groups[val_ids]))
    assert not (set(groups[test_idx])&set(groups[train_ids]))
    assert not (set(groups[test_idx])&set(groups[val_ids]))
    for c in range(6):
        assert any(c in image_classes[i] for i in train_ids),f'Fold {j} TRAIN missing {CLASS_NAMES[c]}'
        assert any(c in image_classes[i] for i in val_ids),f'Fold {j} VAL missing {CLASS_NAMES[c]}; cannot claim six-class 10-fold CV.'
    folds.append((train_ids,val_ids))
print('group_basis:',GROUP_BASIS,'source groups:',len(set(groups)),'real test images:',len(test_idx))
print('Fold lengths:',[(len(tr),len(va)) for tr,va in folds])

# Final training split, based on DEV only. The final test is never read during training or tuning.
final_candidates=GroupShuffleSplit(n_splits=30,test_size=.1,random_state=SEED+2).split(dev_idx,proxies[dev_idx],groups[dev_idx])
final_parts=[]
for tr,va in final_candidates:
    tr_i,va_i=dev_idx[tr],dev_idx[va]
    if all(any(c in image_classes[i] for i in va_i) and any(c in image_classes[i] for i in tr_i) for c in range(6)):
        final_parts.append((tr_i,va_i))
assert final_parts,'Cannot make source-disjoint final train/validation with all classes.'
final_train,final_val=final_parts[0]
assert not set(groups[final_train])&set(groups[final_val])
assert not set(groups[dev_idx])&set(groups[test_idx])

split_manifest={'group_basis':GROUP_BASIS,'class_names':CLASS_NAMES,'seed':SEED,
    'test':[records[i]['relative'] for i in test_idx],
    'folds':[{'fold':j,'train':[records[i]['relative'] for i in tr], 'val':[records[i]['relative'] for i in va]}
             for j,(tr,va) in enumerate(folds,1)],
    'final_train':[records[i]['relative'] for i in final_train],
    'final_val':[records[i]['relative'] for i in final_val]}
(OUT/'split_manifest.json').write_text(json.dumps(split_manifest,indent=2))

## 7. Prepare YOLO dataset YAML for any fold without copying/mutating originals

In [ ]:
def prepare_yolo_split(train_ids,val_ids,test_ids,dirpath):
    dirpath=Path(dirpath);dirpath.mkdir(parents=True,exist_ok=True)
    for name,ids in [('train',train_ids),('val',val_ids),('test',test_ids)]:
        (dirpath/f'{name}.txt').write_text('\n'.join(records[int(i)]['image'] for i in ids)+'\n')
    config={'train':str((dirpath/'train.txt').resolve()),
            'val':str((dirpath/'val.txt').resolve()),
            'test':str((dirpath/'test.txt').resolve()),
            'names':{i:CLASS_NAMES[i] for i in range(6)},'nc':6}
    (dirpath/'data.yaml').write_text(yaml.safe_dump(config,sort_keys=False))
    return dirpath/'data.yaml'

pilot_data_yaml=prepare_yolo_split(*folds[0],test_idx,OUT/'fold01'/'data_files')
print(pilot_data_yaml.read_text()[:1000])

## 8. Define frozen spatial ResNet18 backbone and four head architectures

In [ ]:
from torch import nn
from torch.utils.data import Dataset,DataLoader,TensorDataset,WeightedRandomSampler
from torchvision import transforms as T
import torch.nn.functional as F
IMAGENET_MEAN=torch.tensor((.485,.456,.406),device=DEVICE).view(1,3,1,1)
IMAGENET_STD=torch.tensor((.229,.224,.225),device=DEVICE).view(1,3,1,1)
TF=T.Compose([T.Resize((224,224)),T.ToTensor(),T.Normalize((.485,.456,.406),(.229,.224,.225))])

def new_extractor():
    backbone=timm.create_model('resnet18',pretrained=True,num_classes=0).to(DEVICE).eval()
    for w in backbone.parameters():w.requires_grad_(False)
    return backbone

def encode_tokens(extractor,images):
    fm=extractor.forward_features(images)
    assert fm.ndim==4 and fm.shape[1]==512,fm.shape
    return F.adaptive_avg_pool2d(fm,(4,4)).flatten(2).transpose(1,2)

class CNNHead(nn.Module):
    def __init__(self):
        super().__init__();self.conv=nn.Sequential(nn.Conv1d(512,128,3,padding=1),nn.ReLU(),nn.Conv1d(128,64,3,padding=1),nn.ReLU());self.fc=nn.Sequential(nn.LayerNorm(64),nn.Linear(64,6))
    def forward(self,x):return self.fc(self.conv(x.transpose(1,2)).mean(-1))

class LSTMHead(nn.Module):
    def __init__(self,bi=False,attention=False):
        super().__init__();self.rnn=nn.LSTM(512,64,2,batch_first=True,bidirectional=bi,dropout=.2)
        width=128 if bi else 64
        self.att=nn.Sequential(nn.Linear(width,64),nn.Tanh(),nn.Linear(64,1)) if attention else None
        self.fc=nn.Sequential(nn.LayerNorm(width),nn.Linear(width,32),nn.ReLU(),nn.Dropout(.2),nn.Linear(32,6))
    def forward(self,x):
        z,_=self.rnn(x)
        z=(z*self.att(z).softmax(dim=1)).sum(1) if self.att else z[:,-1]
        return self.fc(z)

class TransformerHead(nn.Module):
    def __init__(self):
        super().__init__();self.embed=nn.Linear(512,128);self.pos=nn.Parameter(torch.zeros(1,16,128))
        self.encoder=nn.TransformerEncoder(nn.TransformerEncoderLayer(128,4,dim_feedforward=256,dropout=.1,batch_first=True),num_layers=2,enable_nested_tensor=False)
        self.fc=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,6))
    def forward(self,x):return self.fc(self.encoder(self.embed(x)+self.pos).mean(1))

METHODS=['CNN','LSTM','Transformer','BiLSTM_Attention','cGAN_Transformer']
def make_head(name):
    if name=='CNN':return CNNHead()
    if name=='LSTM':return LSTMHead()
    if name=='BiLSTM_Attention':return LSTMHead(bi=True,attention=True)
    if name in ('Transformer','cGAN_Transformer'):return TransformerHead()
    raise ValueError(name)

ex=new_extractor()
if (OUT/'resnet18_frozen.pt').exists():
    ex.load_state_dict(torch.load(OUT/'resnet18_frozen.pt',map_location=DEVICE,weights_only=True))
    print('Loaded identical frozen ResNet18 weights from earlier run.')
with torch.inference_mode():
    test_shape=encode_tokens(ex,torch.randn(2,3,224,224,device=DEVICE)).shape
assert test_shape==(2,16,512),test_shape
for m in METHODS:assert make_head(m)(torch.randn(2,16,512)).shape==(2,6)
print('Architectures checked: spatial token tensor',test_shape)

## 9. Extract and CACHE frozen ResNet18 tokens from REAL ground-truth crops

In [ ]:
class ObjectCrops(Dataset):
    def __init__(self,object_indices):self.ids=list(map(int,object_indices))
    def __len__(self):return len(self.ids)
    def __getitem__(self,j):
        ob=objects[self.ids[j]]
        im=Image.open(records[ob['image_idx']]['image']).convert('RGB')
        w0,h0=im.size
        x,y,w,h=ob['bbox']
        left=max(0,math.floor((x-w/2)*w0));right=min(w0,math.ceil((x+w/2)*w0))
        top=max(0,math.floor((y-h/2)*h0));bottom=min(h0,math.ceil((y+h/2)*h0))
        if right<=left or bottom<=top: raise ValueError(f'Degenerate object crop: {ob}')
        return TF(im.crop((left,top,right,bottom))),ob['class_id']

fingerprint=hashlib.sha256(json.dumps({'images':[(r['relative'],r['n_objects']) for r in records],
        'classes':CLASS_NAMES},sort_keys=True).encode()).hexdigest()[:16]
cache_dir=OUT/'resnet_cache_'+fingerprint;cache_dir.mkdir(parents=True,exist_ok=True)
chunk_size=4096;real_feat_blocks=[];real_labels=[]
for start in range(0,len(objects),chunk_size):
    block=cache_dir/f'block_{start:08d}.pt'
    if not block.exists():
        result=[];labels=[]
        loader=DataLoader(ObjectCrops(range(start,min(start+chunk_size,len(objects)))),batch_size=64,num_workers=2,pin_memory=True)
        with torch.inference_mode():
            for xs,ys in tqdm(loader,desc=f'Frozen features {start}',leave=False):
                with torch.autocast('cuda'):
                    feat=encode_tokens(ex,xs.to(DEVICE,non_blocking=True))
                result.append(feat.half().cpu());labels.append(ys.long().cpu())
        torch.save({'features':torch.cat(result),'labels':torch.cat(labels)},block)
    z=torch.load(block,map_location='cpu',weights_only=True)
    real_feat_blocks.append(z['features']);real_labels.append(z['labels'])
    print(f'cached {min(start+chunk_size,len(objects))}/{len(objects)} objects',end='\r')
X=torch.cat(real_feat_blocks);Y=torch.cat(real_labels)
real_feat_blocks.clear();real_labels.clear()
assert X.shape==(len(objects),16,512) and Y.shape==(len(objects),)
obj_image_idx=np.array([ob['image_idx'] for ob in objects],dtype=np.int64)
print('\nCached REAL tokens:',X.shape,'on CPU (float16).')
# Save standalone frozen feature-extractor weights for inference (never trained).
extractor_path=OUT/'resnet18_frozen.pt'
if not extractor_path.exists():torch.save({k:v.cpu() for k,v in ex.state_dict().items()},extractor_path)

## 10. Conditional GAN — define genuinely class-conditional 64×64 crop generator (fold-training data only)

In [ ]:
GAN_TF=T.Compose([T.Resize((64,64)),T.ToTensor(),T.Normalize((.5,.5,.5),(.5,.5,.5))])
class GANCrops(Dataset):
    def __init__(self,ob_ids):self.ob_ids=np.asarray(ob_ids)
    def __len__(self):return len(self.ob_ids)
    def __getitem__(self,k):
        ob=objects[int(self.ob_ids[k])]
        with Image.open(records[ob['image_idx']]['image']) as raw: im=raw.convert('RGB')
        w0,h0=im.size;x,y,w,h=ob['bbox']
        l=max(0,math.floor((x-w/2)*w0));r=min(w0,math.ceil((x+w/2)*w0))
        t=max(0,math.floor((y-h/2)*h0));b=min(h0,math.ceil((y+h/2)*h0))
        assert r>l and b>t
        return GAN_TF(im.crop((l,t,r,b))),ob['class_id']

class ConditionalGenerator(nn.Module):
    def __init__(self,z_dim=128):
        super().__init__();self.z_dim=z_dim;self.emb=nn.Embedding(6,32)
        self.fc=nn.Sequential(nn.Linear(z_dim+32,256*4*4),nn.ReLU())
        self.net=nn.Sequential(
            nn.ConvTranspose2d(256,128,4,2,1),nn.BatchNorm2d(128),nn.ReLU(),
            nn.ConvTranspose2d(128,64,4,2,1),nn.BatchNorm2d(64),nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1),nn.BatchNorm2d(32),nn.ReLU(),
            nn.ConvTranspose2d(32,3,4,2,1),nn.Tanh())
    def forward(self,z,y):
        return self.net(self.fc(torch.cat([z,self.emb(y)],dim=1)).reshape(-1,256,4,4))

class ConditionalDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(9,64,4,2,1),nn.LeakyReLU(.2),
            nn.Conv2d(64,128,4,2,1),nn.BatchNorm2d(128),nn.LeakyReLU(.2),
            nn.Conv2d(128,256,4,2,1),nn.BatchNorm2d(256),nn.LeakyReLU(.2),
            nn.Conv2d(256,512,4,2,1),nn.BatchNorm2d(512),nn.LeakyReLU(.2),nn.Flatten(),nn.Linear(512*4*4,1))
    def forward(self,x,y):
        onehot=F.one_hot(y,6).float().reshape(-1,6,1,1).expand(-1,-1,64,64)
        return self.net(torch.cat([x,onehot],dim=1)).flatten()

def train_fold_gan(training_object_ids,save_dir):
    save_dir=Path(save_dir);save_dir.mkdir(parents=True,exist_ok=True)
    state=save_dir/'cgan_G.pt'
    per_fold_seed=SEED+int(hashlib.sha256(str(save_dir).encode()).hexdigest()[:6],16)
    random.seed(per_fold_seed);np.random.seed(per_fold_seed % (2**32));torch.manual_seed(per_fold_seed);torch.cuda.manual_seed_all(per_fold_seed)
    G=ConditionalGenerator().to(DEVICE)
    if state.exists():
        G.load_state_dict(torch.load(state,map_location=DEVICE,weights_only=True));G.eval();return G
    D=ConditionalDiscriminator().to(DEVICE)
    opt_g=torch.optim.Adam(G.parameters(),lr=.0002,betas=(.5,.999))
    opt_d=torch.optim.Adam(D.parameters(),lr=.0002,betas=(.5,.999))
    subset=GANCrops(training_object_ids)
    classes=np.array([objects[int(i)]['class_id'] for i in training_object_ids])
    freq=np.bincount(classes,minlength=6)
    sample_weights=torch.tensor([1.0/max(freq[c],1) for c in classes],dtype=torch.double)
    sampler=WeightedRandomSampler(sample_weights,num_samples=GAN_STEPS_PER_EPOCH*GAN_BATCH,replacement=True)
    loader=DataLoader(subset,batch_size=GAN_BATCH,sampler=sampler,num_workers=2,pin_memory=True)
    criterion=nn.BCEWithLogitsLoss()
    for ep in range(GAN_EPOCHS):
        G.train();D.train();logs=[]
        for real,cls in loader:
            real=real.to(DEVICE);cls=cls.long().to(DEVICE);bs=len(cls)
            z=torch.randn(bs,128,device=DEVICE)
            fake=G(z,cls)
            loss_d=criterion(D(real,cls),torch.ones(bs,device=DEVICE)) + criterion(D(fake.detach(),cls),torch.zeros(bs,device=DEVICE))
            opt_d.zero_grad();loss_d.backward();opt_d.step()
            z=torch.randn(bs,128,device=DEVICE)
            loss_g=criterion(D(G(z,cls),cls),torch.ones(bs,device=DEVICE))
            opt_g.zero_grad();loss_g.backward();opt_g.step()
            logs.append((float(loss_d.detach()),float(loss_g.detach())))
        print(f' cGAN epoch {ep+1}/{GAN_EPOCHS}: D {np.mean(logs,axis=0)[0]:.3f}; G {np.mean(logs,axis=0)[1]:.3f}',flush=True)
    torch.save({k:v.cpu() for k,v in G.state_dict().items()},state)
    G.eval()
    return G

@torch.inference_mode()
def build_synthetic_features(G,train_obj_ids,extractor,save_dir):
    # Cap is explicit: balancing is attempted on head-training crops only.
    counts_before=np.bincount([objects[int(i)]['class_id'] for i in train_obj_ids],minlength=6)
    target=int(np.quantile(counts_before,GAN_TARGET_QUANTILE))
    wanted=[max(0,target-int(n)) if GAN_MAX_NEW_PER_CLASS is None else min(max(0,target-int(n)),GAN_MAX_NEW_PER_CLASS) for n in counts_before]
    print('GAN real class counts:',counts_before,'target:',target,'requested new:',wanted)
    generated=[];labels=[];rejection=collections.Counter();thumbs=[]
    G.eval();extractor.eval()
    for cls,n_needed in enumerate(wanted):
        accepted=0;attempts=0;fingerprints=set()
        while accepted<n_needed and attempts<max(50,n_needed*5):
            n=min(32,n_needed-accepted,max(1,max(50,n_needed*5)-attempts))
            fake=G(torch.randn(n,128,device=DEVICE),torch.full((n,),cls,device=DEVICE,dtype=torch.long))
            attempts+=n
            pixels=(fake+1)/2
            for im in pixels:
                if float(im.var())<.008:
                    rejection['low_variance']+=1;continue
                preview=(F.interpolate(im.unsqueeze(0),size=(16,16),mode='bilinear',align_corners=False).mul(255).byte().cpu().numpy()).tobytes()
                ph=hashlib.sha1(preview).hexdigest()
                if ph in fingerprints:
                    rejection['identical_generated']+=1;continue
                fingerprints.add(ph)
                x=F.interpolate(im.unsqueeze(0),size=(224,224),mode='bilinear',align_corners=False)
                x=(x-IMAGENET_MEAN)/IMAGENET_STD
                with torch.autocast('cuda'):
                    feat=encode_tokens(extractor,x)
                generated.append(feat.half().cpu());labels.append(cls)
                if len(thumbs)<24:thumbs.append((im.mul(255).byte().permute(1,2,0).cpu().numpy(),cls))
                accepted+=1
                if accepted==n_needed:break
        if accepted<n_needed:print(f'WARNING: generated {accepted}/{n_needed} for {CLASS_NAMES[cls]}')
    new_x=torch.cat(generated) if generated else torch.empty((0,16,512),dtype=torch.float16)
    new_y=torch.tensor(labels,dtype=torch.long)
    if thumbs:
        fig,axs=plt.subplots(4,6,figsize=(12,8))
        for ax,(im,cl) in zip(axs.flat,thumbs):ax.imshow(im);ax.set_title(CLASS_NAMES[cl],fontsize=8);ax.axis('off')
        for ax in axs.flat[len(thumbs):]:ax.axis('off')
        plt.tight_layout();fig.savefig(Path(save_dir)/'cgan_quality_samples.png',dpi=150);plt.close(fig)
    info={'real_counts':counts_before.tolist(),'target':target,'requested_synthetic':wanted,
          'accepted_synthetic':np.bincount(new_y.numpy(),minlength=6).tolist(),
          'after_counts':(counts_before+np.bincount(new_y.numpy(),minlength=6)).tolist(),
          'rejections':dict(rejection),'limitations':'Variance + exact-thumbnail filtering are NOT semantic realism checks. Inspect generated sample images. No claim of perfect balancing.'}
    (Path(save_dir)/'cgan_counts.json').write_text(json.dumps(info,indent=2))
    torch.save({'features':new_x,'labels':new_y},Path(save_dir)/'cgan_tokens.pt')
    return new_x,new_y,info

## 11. Head training, ground-truth crop scores, and correctly matched end-to-end detection scores

- **Crop accuracy/macro-F1:** the head alone on REAL ground-truth object crops. Not an end-to-end score.
- **Hybrid object precision/recall/macro-F1 at fixed IoU and confidence:** YOLO12 boxes → classify detected crops → greedy class-agnostic IoU matching to all real annotated objects. A misclassification contributes both false-positive and false-negative counts. Unmatched ground-truth objects are misses; unmatched detections are false positives. The seventh confusion-matrix label, `background`, records misses/extras.
- **YOLO mAP50 and mAP50-95:** computed separately using Ultralytics validation (do not call fixed-threshold hybrid F1 “mAP”).

In [ ]:
def train_classifier(name,train_object_ids,val_object_ids,save_to,synthetic=None):
    save_to=Path(save_to);save_to.parent.mkdir(parents=True,exist_ok=True)
    random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
    model=make_head(name).to(DEVICE)
    if save_to.exists():
        model.load_state_dict(torch.load(save_to,weights_only=True,map_location=DEVICE));model.eval();return model
    xs=X[train_object_ids];ys=Y[train_object_ids]
    if synthetic is not None and synthetic[0].numel():
        xs=torch.cat([xs,synthetic[0]]);ys=torch.cat([ys,synthetic[1]])
    loader=DataLoader(TensorDataset(xs,ys),batch_size=HEAD_BATCH,shuffle=True,pin_memory=True,num_workers=0)
    optimizer=torch.optim.AdamW(model.parameters(),lr=HEAD_LR,weight_decay=.01)
    hist=[]
    for ep in range(HEAD_EPOCHS):
        model.train();total=0
        for xx,yy in loader:
            xx=xx.float().to(DEVICE,non_blocking=True);yy=yy.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss=F.cross_entropy(model(xx),yy)
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1)
            optimizer.step();total+=float(loss.detach())
        hist.append(total/max(1,len(loader)))
        if ep==0 or (ep+1)%5==0:print(name,f'epoch {ep+1}/{HEAD_EPOCHS} loss {hist[-1]:.4f}',flush=True)
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},save_to)
    (save_to.parent/(save_to.stem+'_loss.json')).write_text(json.dumps(hist))
    model.eval();return model

@torch.inference_mode()
def crop_classification_metrics(head,idxs):
    yt=Y[idxs].numpy();pred=[]
    for ix in range(0,len(idxs),HEAD_BATCH):
        batch=X[idxs[ix:ix+HEAD_BATCH]].float().to(DEVICE)
        pred.extend(head(batch).argmax(1).cpu().tolist())
    pr,re,f1,_=precision_recall_fscore_support(yt,pred,labels=list(range(6)),average='macro',zero_division=0)
    return {'accuracy':float(accuracy_score(yt,pred)),'macro_precision':float(pr),'macro_recall':float(re),'macro_f1':float(f1),
            'confusion_matrix':confusion_matrix(yt,pred,labels=list(range(6))).tolist(),'n_real_crops':len(idxs)}

def boxes_from_gt(image_id):
    out=[]
    with Image.open(records[int(image_id)]['image']) as im:w,h=im.size
    for ob in object_by_image[int(image_id)]:
        cx,cy,bw,bh=ob['bbox']
        out.append((np.array([(cx-bw/2)*w,(cy-bh/2)*h,(cx+bw/2)*w,(cy+bh/2)*h],dtype=float),ob['class_id']))
    return out

def iou_xyxy(a,b):
    inter=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
    area_a=max(0,a[2]-a[0])*max(0,a[3]-a[1]);area_b=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/(area_a+area_b-inter+1e-9)

def accumulate_matches(cm,gt,pred):
    # match by IoU, not by true class; classification errors are counted honestly.
    options=sorted([(iou_xyxy(gb,pb),gi,pi) for gi,(gb,_) in enumerate(gt)
                   for pi,(pb,_,_) in enumerate(pred)],reverse=True)
    used_g=set();used_p=set()
    for iou,gi,pi in options:
        if iou<MATCH_IOU:break
        if gi in used_g or pi in used_p:continue
        cm[gt[gi][1],pred[pi][1]]+=1;used_g.add(gi);used_p.add(pi)
    for gi,(_,c) in enumerate(gt):
        if gi not in used_g:cm[c,6]+=1
    for pi,(_,c,_) in enumerate(pred):
        if pi not in used_p:cm[6,c]+=1

def summarize_matching(cm):
    tp=np.diag(cm)[:6].astype(float)
    fp=cm[:, :6].sum(axis=0)-tp
    fn=cm[:6, :].sum(axis=1)-tp
    prec=np.divide(tp,tp+fp,out=np.zeros(6),where=(tp+fp)>0)
    rec=np.divide(tp,tp+fn,out=np.zeros(6),where=(tp+fn)>0)
    f1=np.divide(2*prec*rec,prec+rec,out=np.zeros(6),where=(prec+rec)>0)
    return {'precision_macro':float(prec.mean()),'recall_macro':float(rec.mean()),'f1_macro':float(f1.mean()),
           'precision_by_class':prec.tolist(),'recall_by_class':rec.tolist(),'f1_by_class':f1.tolist(),
           'true_positives':int(tp.sum()),'false_positives':int(fp.sum()),'false_negatives':int(fn.sum()),
           'confusion_matrix_with_background':cm.tolist()}

@torch.inference_mode()
def evaluate_end_to_end(detector,heads,ids,extractor,tag):
    matrices={'YOLO12':np.zeros((7,7),dtype=np.int64)}
    for name in heads:matrices[name]=np.zeros((7,7),dtype=np.int64)
    # IMPORTANT: batch image paths in groups; passing all 5000 paths to predict(list) can
    # cause Ultralytics to preload all images and run out of Colab RAM.
    iterator=range(0,len(ids),16)
    for start in tqdm(iterator,total=math.ceil(len(ids)/16),desc=tag):
        image_ids=ids[start:start+16]
        paths=[records[int(k)]['image'] for k in image_ids]
        itr=detector.predict(source=paths,conf=DETECT_CONF,iou=.7,imgsz=YOLO_IMGSZ,
                             device=0,stream=True,save=False,verbose=False)
        for image_id,res in zip(image_ids,itr):
            gt=boxes_from_gt(image_id)
            pb=res.boxes.xyxy.detach().cpu().numpy()
            pc=res.boxes.cls.int().detach().cpu().numpy()
            ps=res.boxes.conf.detach().cpu().numpy()
            base=[(box,int(cls),float(score)) for box,cls,score in zip(pb,pc,ps)]
            accumulate_matches(matrices['YOLO12'],gt,base)
            if not heads or len(pb)==0:
                for name in heads:accumulate_matches(matrices[name],gt,[])
                continue
            with Image.open(records[int(image_id)]['image']) as src:im=src.convert('RGB')
            patches=[];valid=[]
            for j,box in enumerate(pb):
                l,t,r,b=[int(round(z)) for z in box]
                l=max(0,min(l,im.width));r=max(0,min(r,im.width))
                t=max(0,min(t,im.height));b=max(0,min(b,im.height))
                if r>l and b>t:patches.append(TF(im.crop((l,t,r,b))));valid.append(j)
            if patches:
                with torch.autocast('cuda'):
                    features=encode_tokens(extractor,torch.stack(patches).to(DEVICE)).float()
            else:features=None
            for name,head in heads.items():
                cls_pred=np.full(len(pb),-1,dtype=int)
                if features is not None:
                    cls_pred[valid]=head(features).argmax(1).cpu().numpy()
                predictions=[(pb[j],int(cls_pred[j]),float(ps[j])) for j in range(len(pb)) if cls_pred[j]>=0]
                accumulate_matches(matrices[name],gt,predictions)
    return {k:summarize_matching(v) for k,v in matrices.items()}

object_by_image=[[] for _ in records]
for ob in objects:object_by_image[ob['image_idx']].append(ob)

def object_indices_for_images(image_ids):
    return np.flatnonzero(np.isin(obj_image_idx,np.asarray(image_ids)))

assert iou_xyxy(np.array([0,0,10,10]),np.array([0,0,10,10]))>.99
fake_cm=np.zeros((7,7),dtype=np.int64)
accumulate_matches(fake_cm,[(np.array([0,0,10,10]),2)],[(np.array([0,0,10,10]),1,.9)])
assert fake_cm[2,1]==1 and summarize_matching(fake_cm)['true_positives']==0
print('End-to-end matching sanity checks passed.')

## 12. Run manifest, resumable 10-fold YOLO12 + all recurrent/attention/cGAN experiments

In [ ]:
config_record={
 'dataset_fingerprint':fingerprint,'group_basis':GROUP_BASIS,'names':CLASS_NAMES,
 'yolo_weights':YOLO_WEIGHTS,'yolo_imgsz':YOLO_IMGSZ,'yolo_epochs':YOLO_EPOCHS,
 'head_epochs':HEAD_EPOCHS,'gan_epochs':GAN_EPOCHS,'gan_steps':GAN_STEPS_PER_EPOCH,
 'gan_max_new_per_class':GAN_MAX_NEW_PER_CLASS,'gan_target_quantile':GAN_TARGET_QUANTILE,
 'seed':SEED,'num_folds':NUM_FOLDS,'smoke':SMOKE_TEST,
 'sample_min_images_per_class':IMAGE_SAMPLES_PER_CLASS,
 'extra_background_images':EXTRA_BACKGROUND_IMAGES,
 'yolo_batch':YOLO_BATCH,'head_batch':HEAD_BATCH,'gan_batch':GAN_BATCH,
 'torch':torch.__version__,'ultralytics':ultralytics.__version__,'timm':timm.__version__}
run_hash=hashlib.sha256(json.dumps(config_record,sort_keys=True).encode()).hexdigest()[:12]
RUN=OUT/('deadline_pilot_' if SMOKE_TEST else 'deadline_').__add__(run_hash)
RUN.mkdir(exist_ok=True,parents=True)
(RUN/'run_config.json').write_text(json.dumps(config_record,indent=2))
print('NEW reduced-subset run:',RUN)
print('Original full-size run is unaffected.')

def train_detector(data_yaml,folder):
    folder=Path(folder)
    best=folder/'weights'/'best.pt'
    done=folder/'detector_finished.txt'
    if done.exists() and best.exists():return YOLO(str(best))
    last=folder/'weights'/'last.pt'
    if last.exists():
        print('Resuming interrupted YOLO train from',last)
        model=YOLO(str(last));model.train(resume=True)
    else:
        model=YOLO(YOLO_WEIGHTS)
        model.train(data=str(data_yaml),epochs=YOLO_EPOCHS,imgsz=YOLO_IMGSZ,batch=YOLO_BATCH,
                    device=0,workers=2,patience=YOLO_PATIENCE,seed=SEED,
                    project=str(folder.parent),name=folder.name,exist_ok=True,
                    cache=False,plots=True,verbose=False)
    assert best.exists(),f'YOLO training did not produce best.pt: {best}'
    done.write_text('complete')
    return YOLO(str(best))

def augment_tokens_only(train_obids,fold_dir):
    synth_file=Path(fold_dir)/'cgan_tokens.pt'
    if synth_file.exists():
        stuff=torch.load(synth_file,weights_only=True,map_location='cpu')
        info=json.loads((Path(fold_dir)/'cgan_counts.json').read_text())
        return (stuff['features'],stuff['labels']),info
    G=train_fold_gan(train_obids,fold_dir)
    sx,sy,info=build_synthetic_features(G,train_obids,ex,fold_dir)
    return (sx,sy),info

all_fold_summaries=[]
folds_to_run=folds[:1] if SMOKE_TEST else folds
for fold_num,(tr_imgs,va_imgs) in enumerate(folds_to_run,1):
    fld=RUN/f'fold_{fold_num:02d}'
    fld.mkdir(exist_ok=True,parents=True)
    result_path=fld/'fold_metrics.json'
    if result_path.exists():
        print('Skipping completed fold',fold_num)
        all_fold_summaries.append(json.loads(result_path.read_text()))
        continue
    print('\n========= FOLD',fold_num,'/10 =========',flush=True)
    d_yaml=prepare_yolo_split(tr_imgs,va_imgs,test_idx,fld/'data')
    train_o=object_indices_for_images(tr_imgs);val_o=object_indices_for_images(va_imgs)
    assert len(train_o) and len(val_o)
    detector=train_detector(d_yaml,fld/'yolo12')
    box=detector.val(data=str(d_yaml),split='val',imgsz=YOLO_IMGSZ,device=0,plots=False,verbose=False)
    detector_metrics={'mAP50':float(box.box.map50),'mAP50_95':float(box.box.map),'precision':float(box.box.mp),'recall':float(box.box.mr)}
    (fld/'detector_metrics.json').write_text(json.dumps(detector_metrics,indent=2))
    # GAN is fitted ONLY on current fold TRAIN object crops; CV validation/test never touches GAN training.
    synth,cgan_info=augment_tokens_only(train_o,fld/'cgan')
    heads={}
    crop_scores={}
    for name in METHODS:
        synth_arg=synth if name=='cGAN_Transformer' else None
        head=train_classifier(name,train_o,val_o,fld/'heads'/f'{name}.pt',synthetic=synth_arg)
        crop_scores[name]=crop_classification_metrics(head,val_o)
        heads[name]=head
        (fld/'crop_scores_progress.json').write_text(json.dumps(crop_scores,indent=2))
    e2e=evaluate_end_to_end(detector,heads,va_imgs,ex,f'Fold {fold_num} REAL detector→head validation') if RUN_END_TO_END_EACH_FOLD else {}
    summary={'fold':fold_num,'n_train_images':len(tr_imgs),'n_validation_images':len(va_imgs),
             'n_train_real_objects':len(train_o),'n_val_real_objects':len(val_o),
             'yolo_detector':detector_metrics,'crop_classifiers':crop_scores,
             'end_to_end':e2e,'gan':cgan_info}
    result_path.write_text(json.dumps(summary,indent=2))
    all_fold_summaries.append(summary)
    del heads,detector
    torch.cuda.empty_cache()
    (RUN/'cv_progress.json').write_text(json.dumps(all_fold_summaries,indent=2))
assert SMOKE_TEST or len(all_fold_summaries)==10,'Not all 10 folds are complete.'
print('Completed folds:',len(all_fold_summaries))

## 13. Ten-fold numerical summary and visual comparisons (no pilot presented as full results)

**Deadline note:** Detector and crop-head scores are different metrics. Use the end-to-end Fold 1 preview for an apples-to-apples single-fold comparison; do not present it as 10-fold E2E validation.

In [ ]:
assert all_fold_summaries
summary_rows=[]
for name in ['YOLO12']+METHODS:
    if name=='YOLO12':
        values=[r['yolo_detector']['mAP50_95'] for r in all_fold_summaries]
        metric='YOLO mAP50-95'
    else:
        values=[r['crop_classifiers'][name]['macro_f1'] for r in all_fold_summaries]
        metric='REAL GT crop macro-F1'
    summary_rows.append({'model':name,'metric':metric,'n_folds':len(values),'mean':float(np.mean(values)),
                         'sample_std':float(np.std(values,ddof=1)) if len(values)>1 else None})
cv_summary=pd.DataFrame(summary_rows)
print('PILOT ONLY — NOT TEN-FOLD RESULTS' if SMOKE_TEST else 'FULL TEN-FOLD CV')
display(cv_summary)
cv_summary.to_csv(RUN/'cv_summary.csv',index=False)
if not SMOKE_TEST:
    fig,ax=plt.subplots(figsize=(10,5))
    ax.bar(cv_summary['model'],cv_summary['mean'])
    ax.set_ylim(0,1);ax.set_ylabel('Reported score (different metric for YOLO)');ax.set_title('CV scores — YOLO mAP vs head crop F1 (not directly comparable)')
    plt.xticks(rotation=20,ha='right');plt.tight_layout();fig.savefig(RUN/'cv_scores_noncomparable.png',dpi=150)
    plt.show()
if RUN_END_TO_END_EACH_FOLD:
    joint=[]
    for name in ['YOLO12']+METHODS:
        values=[r['end_to_end'][name]['f1_macro'] for r in all_fold_summaries]
        joint.append({'model':name,'mean_end_to_end_f1':float(np.mean(values)),
                      'std_end_to_end_f1':float(np.std(values,ddof=1)) if len(values)>1 else None})
    e2e_frame=pd.DataFrame(joint)
    display(e2e_frame)
    e2e_frame.to_csv(RUN/'cv_end_to_end_f1.csv',index=False)
    if not SMOKE_TEST:
        fig,ax=plt.subplots(figsize=(10,5));ax.bar(e2e_frame.model,e2e_frame.mean_end_to_end_f1)
        ax.set_ylim(0,1);ax.set_ylabel('IoU=0.5 matched macro-F1, detector→classifier');ax.set_title('Comparable end-to-end CV metric')
        plt.xticks(rotation=20,ha='right');plt.tight_layout();fig.savefig(RUN/'cv_end_to_end_f1.png',dpi=150);plt.show()

## 13A. Optional Fold 1 end-to-end comparison + deployable demo (no extra detector training)

Runs real detector→head validation **on Fold 1 only**, so figures are **not** 10-fold end-to-end cross-validation. Saves a demo package from Fold 1 checkpoints, not a final model trained on all development data. A separate locked test is still required for final generalization claims. Set `RUN_FOLD1_E2E_PREVIEW=False` in config to skip this.

In [ ]:
if RUN_FOLD1_E2E_PREVIEW and not SMOKE_TEST:
    preview_path=RUN/'fold_01'/'fold1_end_to_end_preview.json'
    if preview_path.exists():
        preview_results=json.loads(preview_path.read_text())
        print('Loaded existing Fold 1 preview')
    else:
        preview_folder=RUN/'fold_01'
        detector=YOLO(str(preview_folder/'yolo12'/'weights'/'best.pt'))
        heads={}
        for method in METHODS:
            head=make_head(method).to(DEVICE)
            head.load_state_dict(torch.load(preview_folder/'heads'/f'{method}.pt',weights_only=True,map_location=DEVICE))
            head.eval(); heads[method]=head
        preview_results=evaluate_end_to_end(detector,heads,folds[0][1],ex,'Fold1 end-to-end PREVIEW')
        preview_path.write_text(json.dumps(preview_results,indent=2))
        del detector,heads
        torch.cuda.empty_cache()
    display(pd.DataFrame([
      {'model':name,'end_to_end_macro_F1_pct':round(val['f1_macro']*100,2)}
      for name,val in preview_results.items()
    ]).sort_values('end_to_end_macro_F1_pct',ascending=False))

    demo=RUN/'fold1_demo_export'
    demo.mkdir(parents=True,exist_ok=True)
    shutil.copy2(RUN/'fold_01'/'yolo12'/'weights'/'best.pt',demo/'yolo12_best.pt')
    shutil.copy2(OUT/'resnet18_frozen.pt',demo/'resnet18_frozen.pt')
    for method in METHODS:
        shutil.copy2(RUN/'fold_01'/'heads'/f'{method}.pt',demo/f'{method}_head.pt')
    (demo/'README.txt').write_text(
        'Fold 1 validation-trained demo checkpoints; NOT a final all-data model. '
        'Never describe this as held-out final test, or full 10-fold E2E validation.\n')
    print('Saved Fold 1 demo package:',demo)
else:
    print('End-to-end preview disabled; 10-fold crop + detector metrics saved as CSV.')

## 14. Final locked holdout experiment

The previously untouched ~10% **real test** partition is used once, after model choices and training budgets are fixed. Final models train only on `final_train` and monitor `final_val` (both from development data). cGAN is fitted only on real `final_train` crops. This final evaluation measures detector performance **and** hybrid object-level performance; no GAN image enters evaluation.

In [ ]:
if RUN_FINAL:
    assert not SMOKE_TEST,'Cannot do final test in pilot.'
    assert len(all_fold_summaries)==10,'Complete all CV folds first.'
    final_dir=RUN/'final';final_dir.mkdir(exist_ok=True,parents=True)
    final_results_file=final_dir/'final_test_metrics.json'
    if final_results_file.exists():
        final_result=json.loads(final_results_file.read_text())
        print('Loaded prior locked test evaluation; avoiding repeated test-based tuning.')
    else:
        dy=prepare_yolo_split(final_train,final_val,test_idx,final_dir/'data')
        final_detector=train_detector(dy,final_dir/'yolo12')
        final_gan_ids=object_indices_for_images(final_train)
        real_val_ids=object_indices_for_images(final_val)
        real_test_ids=object_indices_for_images(test_idx)
        syn_final,gan_final_info=augment_tokens_only(final_gan_ids,final_dir/'cgan')
        final_heads={}
        for name in METHODS:
            final_heads[name]=train_classifier(name,final_gan_ids,real_val_ids,final_dir/'heads'/f'{name}.pt',
                                               synthetic=syn_final if name=='cGAN_Transformer' else None)
        # Final test ONCE. No repeated model decisions based on these scores.
        det_test=final_detector.val(data=str(dy),split='test',imgsz=YOLO_IMGSZ,device=0,plots=True,verbose=False)
        test_box={'mAP50':float(det_test.box.map50),'mAP50_95':float(det_test.box.map),
                  'precision':float(det_test.box.mp),'recall':float(det_test.box.mr)}
        test_crop={name:crop_classification_metrics(head,real_test_ids) for name,head in final_heads.items()}
        test_e2e=evaluate_end_to_end(final_detector,final_heads,test_idx,ex,'LOCKED final test (real images)')
        final_result={'scope':'Once-only real external test. Crop, detector mAP and end-to-end metrics are different.',
                      'group_basis':GROUP_BASIS,'test_real_images':len(test_idx),'test_real_objects':len(real_test_ids),
                      'yolo_detector':test_box,'crop_classifiers':test_crop,'end_to_end':test_e2e,
                      'gan_training_augmentation':gan_final_info}
        final_results_file.write_text(json.dumps(final_result,indent=2))
    print('Final test YOLO12 detector:',final_result['yolo_detector'])
    print('Final detector→head macro-F1 (IoU=.5):')
    for name,metrics in final_result['end_to_end'].items():print(name,round(metrics['f1_macro']*100,2),'%')
else:
    print('RUN_FINAL=False; final test withheld (correct for a pilot).')

## 15. Final confusion matrices, classwise metrics, GAN before/after table

In [ ]:
if RUN_FINAL:
    from matplotlib import pyplot as plt
    results=final_result
    pd.DataFrame({'class':CLASS_NAMES,
                  'real_training_objects':results['gan_training_augmentation']['real_counts'],
                  'synthetic_cgan_crops_added':results['gan_training_augmentation']['accepted_synthetic'],
                  'after_classification_training':results['gan_training_augmentation']['after_counts']}).to_csv(RUN/'gan_before_after.csv',index=False)
    display(pd.read_csv(RUN/'gan_before_after.csv'))
    for model_name,model_metrics in results['end_to_end'].items():
        cm=np.array(model_metrics['confusion_matrix_with_background'])
        fig,ax=plt.subplots(figsize=(7,6))
        mesh=ax.imshow(cm,cmap='Greys');plt.colorbar(mesh,ax=ax)
        labels=CLASS_NAMES+['background']
        ax.set_xticks(range(7),labels,rotation=45,ha='right');ax.set_yticks(range(7),labels)
        ax.set_xlabel('Predicted class / background for misses')
        ax.set_ylabel('Actual class / background for extra detections')
        ax.set_title(model_name+' — real test, matched at IoU ≥ 0.5')
        plt.tight_layout();fig.savefig(RUN/f'{model_name}_final_confusion.png',dpi=160);plt.show()

## 16. Export final YOLO12 and classifier heads for your existing dashboard

In [ ]:
if RUN_FINAL:
    export_dir=RUN/'dashboard_export';export_dir.mkdir(exist_ok=True,parents=True)
    for f in ['resnet18_frozen.pt']:
        shutil.copy2(OUT/f,export_dir/f)
    shutil.copy2(RUN/'final'/'yolo12'/'weights'/'best.pt',export_dir/'yolo12_best.pt')
    for name in METHODS:
        shutil.copy2(RUN/'final'/'heads'/f'{name}.pt',export_dir/f'{name}_head.pt')
    runtime_info={'class_names':CLASS_NAMES,'yolo_model':'yolo12_best.pt','extractor_arch':'timm_resnet18',
                  'extractor_weights':'resnet18_frozen.pt','resnet_input_size':224,
                  'resnet_normalize_mean':[.485,.456,.406],'resnet_normalize_std':[.229,.224,.225],
                  'token_layout':'adaptive_avg_pool2d 4x4, flatten(2), transpose(1,2): [B,16,512]',
                  'classifiers':{name:name+'_head.pt' for name in METHODS},
                  'cv_folds':10,'group_basis':GROUP_BASIS,
                  'notes':'Actual dashboard must crop YOLO-predicted boxes and then run ResNet18+selected classifier.'}
    (export_dir/'inference_metadata.json').write_text(json.dumps(runtime_info,indent=2))
    shutil.copy2(RUN/'final'/'final_test_metrics.json',export_dir/'final_test_metrics.json')
    package=RUN/'hybrid_final_dashboard_package.zip'
    with zipfile.ZipFile(package,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:
        for p in export_dir.rglob('*'):
            if p.is_file():z.write(p,p.relative_to(export_dir.parent))
    print('Saved deployable package:',package)
    print('Saved complete run with CV and test:',RUN)
else:
    print('Export is only available after full 10-fold CV and final model fitting.')

## 17. Simple end-to-end demo: original image → YOLO12 boxes → selected LSTM/BiLSTM/Transformer crop labels

In [ ]:
if RUN_FINAL:
    from PIL import ImageDraw
    chosen_head='BiLSTM_Attention' # also CNN/LSTM/Transformer/cGAN_Transformer
    package=RUN/'dashboard_export'
    det=YOLO(str(package/'yolo12_best.pt'))
    extractor=new_extractor();extractor.load_state_dict(torch.load(package/'resnet18_frozen.pt',map_location=DEVICE,weights_only=True));extractor.eval()
    model=make_head(chosen_head).to(DEVICE)
    model.load_state_dict(torch.load(package/f'{chosen_head}_head.pt',map_location=DEVICE,weights_only=True));model.eval()
    example_path=records[int(test_idx[0])]['image']
    with Image.open(example_path) as im:img=im.convert('RGB')
    inference=det.predict(source=example_path,imgsz=YOLO_IMGSZ,conf=DETECT_CONF,verbose=False)[0]
    draw=ImageDraw.Draw(img)
    with torch.inference_mode():
        for box,confidence in zip(inference.boxes.xyxy.cpu().numpy(),inference.boxes.conf.cpu().numpy()):
            l,t,r,b=[int(round(v)) for v in box]
            l=max(0,l);t=max(0,t);r=min(img.width,r);b=min(img.height,b)
            if r<=l or b<=t:continue
            crop=TF(img.crop((l,t,r,b))).unsqueeze(0).to(DEVICE)
            with torch.autocast('cuda'):tok=encode_tokens(extractor,crop).float()
            probs=model(tok).softmax(1)[0]
            c=int(probs.argmax())
            draw.rectangle((l,t,r,b),outline='yellow',width=3)
            draw.text((l,max(0,t-14)),f'{CLASS_NAMES[c]} head={float(probs[c]):.2f} YOLO={float(confidence):.2f}',fill='yellow')
    display(img)
    print('YOLO class confidence is distinct from classifier-head confidence; not calibrated.')

## 18. Methodology and reporting checklist

- Report *observed* counts from `dataset_audit.json`, not just historical audit numbers. Count both **images** and **labeled objects**. Background frames contain no objects.
- Cite original sources (AOD-4 and MBAOD) and acknowledge uncertainty about original source/sequence grouping if unavailable. Deduplication does **not** prove sequence independence.
- Store/submit `split_manifest.json` and record whether fold grouping was genuinely source-based or the image-level fallback.
- cGAN is a **conditional crop generator**. Its samples may be unrealistic: check `cgan_quality_samples.png`; code-level checks are not a validated realism metric. State class counts before/after rather than asserting full equalization under capped generation.
- The detector is trained on **real YOLO-box-annotated images**. Only the `cGAN_Transformer` classifier head sees generated crop features. No synthetic validation/test data.
- Crop F1 and 10-fold detector mAP are **not directly comparable**. For complete hybrid comparison use `cv_end_to_end_f1.csv` and matching test F1. Report the detector’s mAP separately.
- External test is locked before CV and evaluated only after the ten folds. Pilot is not completed CV.
- These spatial LSTMs do **not** learn motion. The dashboard's ByteTrack module is separate.
- Note GPU type, package versions, model sizes, training epochs, each fold's train/validation counts, per-class support, fold means ± standard deviations, final test classwise results and confusion matrices. Avoid claims of 100% data balance or superiority without supporting measured results.
- Local Windows `E:\AI project\DS\comb DSF` is **not visible in Colab**: upload a zipped deduplicated copy or mount it from Drive.

**Deadline protocol:** At-least-3k-images-containing-each-class sampling is not exact per-class bounding-box balance; include the saved sample audit. Exactly 10 independent YOLO fold trainings (5 epochs each) and 5 heads per fold are performed. Fold1 end-to-end preview only, and no final untouched test was run unless independently executed. Do not borrow old 55k Fold1 metrics into this reduced study.


## 19. Export a complete Fold 1 runtime package without retraining

Run this after the original summary/export cells. It checks the saved files and includes the matching
ResNet18, five heads, configuration and real ten-fold results. This archive is useful for replacing
the YOLO12 model folder in a dashboard. It is **not** the full two-family dashboard bundle.


In [ ]:
from pathlib import Path
import zipfile, shutil
assert 'RUN' in globals(), 'Run the setup/summary cells first so RUN identifies your saved experiment.'
demo=RUN/'fold1_demo_export'
sources={'yolo12_best.pt':RUN/'fold_01/yolo12/weights/best.pt',
         'resnet18_frozen.pt':OUT/'resnet18_frozen.pt'}
sources.update({f'{method}_head.pt':RUN/'fold_01/heads'/f'{method}.pt' for method in METHODS})
sources.update({name:RUN/name for name in ['run_config.json','cv_summary.csv','cv_progress.json']})
missing=[str(p) for p in sources.values() if not p.is_file()]
assert not missing, 'Missing saved files: '+str(missing)
demo.mkdir(parents=True,exist_ok=True)
for name,path in sources.items():
    if path.resolve()!=(demo/name).resolve():shutil.copy2(path,demo/name)
(demo/'README.txt').write_text('Fold 1 demo checkpoints, not final all-data weights. Detector mAP and crop F1 are separate.\n')
archive=RUN/'YOLO12_Fold1_With_Results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(demo.iterdir()):
        if path.is_file():z.write(path,'yolo12_fold1/'+path.name)
print('Saved:',archive)
from google.colab import files
files.download(str(archive))
